# Non-Negative Matrix Factorization (NMF) vs PCA

## Setup
1. Download the file `face.train.tar.gz` from Moodle, and place it in the same directory as this notebook
2.  Ensure you have PyTorch, scikit-learn, and torchvision installed, as we'll use these libraries in this notebook. You can install them using: `pip install torch torchvision scikit-learn`



In [4]:
import glob
import numpy as np
import torch
from PIL import Image

import tarfile

# Extract the .tar.gz file
# Using GDrive:
from google.colab import drive
drive.mount('/content/drive')
dir = '/content/drive/MyDrive/Tutorials/Tutorial_9_coding_materials/face.train.tar.gz'
with tarfile.open(dir, 'r:gz') as tar:
    tar.extractall()

# Prepare the data matrix X
images = []
for file in glob.glob('train/face/*.pgm'):
    images.append(torch.FloatTensor(np.array(Image.open(file))))

# Stack images into a tensor (height, width, num_images)
X = torch.stack(images, dim=-1)
height, width, num_images = X.shape
print(f'Loaded {num_images} images of size {height} x {width}')

img_size = X.shape[:2]
X = X.flatten(0, 1) # Unroll each 2D image into a 1D vector
X /= 255 # Normalize pixel values to range [0, 1]

X.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_4036/3812841674.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Loaded 2429 images of size 19 x 19


torch.Size([361, 2429])

## **NMF**

In [8]:
import torch.optim as optim
import torch.nn as nn
import math

K = ... # TODO - Choose an Appropriate Number of Features
num_steps = 5000

# Initialize U and V from uniform distribution U(0, 1), scaled by 1/sqrt(K) to stabilize initial optimization

U = (torch.rand(X.shape[0], K)  / math.sqrt(K)).requires_grad_()
V = (torch.rand(K, X.shape[1])  / math.sqrt(K)).requires_grad_()

# TODO - Define the optimizer
optimizer = ...
# TODO - Define the loss function
criterion = ...

loss_curve = []
for i in range(num_steps):
    # TODO: Implement the training routine here.
    ...
    loss_val = ... # TODO - store current loss value
    loss_curve.append(loss_val)

    if (i + 1) % 1000 == 0 or i == 0:
        print(f'Step {i+1} | Loss: {loss_val:.05f}')

TypeError: rand(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got ellipsis"

In [ ]:
# Loss curve plot
import matplotlib.pyplot as plt
%matplotlib inline

plt.plot(loss_curve)
plt.title('Loss curve')
plt.xlabel('Iteration')
plt.ylabel('Loss (MSE)')
plt.yscale('log')

In [ ]:
import torchvision
# Each of the K columns in U is a basis feature vector of size 361 (19x19 pixels).

# TODO - Reshape the basis vectors in U into images of size 19x19
features = ...
# TOD0 - Normalize all features globally to [0, 1] for clear visualization
features = ... # Normalize in [0, 1]

# Visualize the basis features in a grid
img = torchvision.utils.make_grid(features.unsqueeze(1), nrow=int(math.sqrt(K)), pad_value=1)
plt.figure(figsize=(6, 6))
plt.imshow(img.permute(1, 2, 0))
plt.axis('off')
plt.title('NMF features')

In [ ]:
image_idx = 0  # Select the index of the image to visualize feature assignments

# TODO - Extract the vector from the assignment matrix V corresponding to the feature assignments for mage_idx
feature_assignment = ...
# TOD0 - Normalize all features globally to [0, 1] for clear visualization
feature_assignment = ...

plt.imshow(feature_assignment.view(-1, int(math.sqrt(K))).unsqueeze(-1).expand(-1, -1, 3))
plt.axis('off')

# PCA

In [ ]:
from sklearn.decomposition import PCA

K = ... # TODO - Choose an Appropriate Number of Features

X_np = X.numpy() # Convert tensor to NumPy for sklearn
# TODO - Apply PCA to the matrix X (Hint: Use the predefined PCA function)
# Hint: You may need to transpose your matrix appropriately,
...
X_reduced = ...
X_reconstructed = ...
# Reconstruction Loss (MSE)
reconstruction_loss = ((X_np - X_reconstructed)**2).mean()

print(f'Reconstruction Loss: {reconstruction_loss:.5f}')

In [ ]:
# Visualization of principal components (Eigenfaces)
# TODO - Retrieve PCA components and convert them to PyTorch tensor
principal_components = ...
features = principal_components.view(-1, *img_size).clone()
# TOD0 - Normalize all features globally to [0, 1] for clear visualization
features = ... # Normalize [0, 1]

img = torchvision.utils.make_grid(features.unsqueeze(1), nrow=int(math.sqrt(K)), pad_value=1)
plt.figure(figsize=(8, 8))
plt.imshow(img.permute(1, 2, 0))
plt.axis('off')
plt.title('Principal Components (Eigenfaces)')
plt.show()

In [ ]:
# Select the image index you want to visualize
image_idx = 3

# PCA component weights for the selected image (analogous to NMF feature assignments)
feature_assignment = torch.from_numpy(X_reduced[:, image_idx])

# TOD0 - Normalize all features globally to [0, 1] for clear visualization
feature_assignment = ...
feature_img_size = int(math.sqrt(K))
feature_assignment_img = feature_assignment.view(feature_img_size, feature_img_size)
feature_assignment_rgb = feature_assignment_img.unsqueeze(-1).expand(-1, -1, 3)

# Plot
plt.imshow(feature_assignment_rgb)
plt.axis('off')
plt.show()

What are the main differences between the features learned by PCA and those learned by NMF?

TODO - Answer

